# Câu 1 - Thuật toán DFS

Tìm đường thoát từ phòng trung tâm ra rìa lâu đài trên ma trận `n x n`. Hai ô chỉ nối với nhau nếu có chung cạnh. Ô có giá trị `1` là đi được, ô có giá trị `0` là không đi được.

File đầu vào dùng trong bài là `A_in.csv`. Notebook này trình bày riêng thuật toán DFS.

## Đề bài và dữ liệu đầu vào

Dòng đầu của `A_in.csv` là:

```text
10,5,5
```

Ý nghĩa:

- `n = 10`: lâu đài là ma trận `10 x 10`.
- Vị trí bắt đầu là `(5,5)`.
- Tọa độ dùng kiểu `0-based`, tức dòng/cột bắt đầu từ `0`.

Mục tiêu là tìm một đường đi từ `(5,5)` đến một ô bất kỳ nằm trên rìa ma trận.

## Biểu diễn bài toán dưới dạng đồ thị

Bài toán lâu đài có thể xem như một bài toán tìm đường trên đồ thị:

- Mỗi ô trong ma trận là một đỉnh.
- Chỉ những ô có giá trị `1` mới là đỉnh hợp lệ vì đó là ô có đường hầm.
- Hai đỉnh có cạnh nối với nhau nếu hai ô tương ứng kề cạnh theo một trong bốn hướng: trên, phải, dưới, trái.
- Ô bắt đầu là `(5,5)`.
- Tập đích là các ô đi được nằm trên rìa ma trận.

Chương trình không tạo sẵn toàn bộ danh sách cạnh. Thay vào đó, khi đang xét một ô, hàm `get_neighbors()` sinh ra các ô kề hợp lệ. Vì vậy đồ thị được suy ra trực tiếp từ ma trận đầu vào.

Ví dụ, từ ô `(5,5)`, chương trình xét các ô `(4,5)`, `(5,6)`, `(6,5)`, `(5,4)`, rồi chỉ giữ lại các ô nằm trong ma trận và có giá trị `1`.

## Code chương trình DFS

Dưới đây là chương trình hiện có trong file `cau1_dfs.py`. Notebook chỉ sao chép nội dung để trình bày, không chỉnh sửa file code gốc.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt


def read_input(file_path):
    with open(file_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        first_line = next(reader)
        n = int(first_line[0])
        start = (int(first_line[1]), int(first_line[2]))
        maze = [[int(value) for value in row] for row in reader]

    return n, start, maze


def is_inside(position, n):
    row, col = position
    return 0 <= row < n and 0 <= col < n


def is_border(position, n):
    row, col = position
    return row == 0 or row == n - 1 or col == 0 or col == n - 1


def get_neighbors(position, n, maze):
    row, col = position
    directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    neighbors = []

    for d_row, d_col in directions:
        next_pos = (row + d_row, col + d_col)

        if is_inside(next_pos, n) and maze[next_pos[0]][next_pos[1]] == 1:
            neighbors.append(next_pos)

    return neighbors


def reconstruct_path(parent, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def dfs_escape(n, start, maze):
    if not is_inside(start, n) or maze[start[0]][start[1]] != 1:
        return None

    stack = [start]
    parent = {start: None}
    visited = {start}

    while stack:
        current = stack.pop()

        if is_border(current, n):
            return reconstruct_path(parent, current)

        for neighbor in reversed(get_neighbors(current, n, maze)):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = current
                stack.append(neighbor)

    return None


def write_output(file_path, path):
    with open(file_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)

        if path is None:
            writer.writerow([-1])
            return

        writer.writerow([len(path)])
        writer.writerows(path)


def save_path_chart(maze, path, output_file):
    n = len(maze)

    plt.figure(figsize=(6, 6))
    plt.imshow(maze, cmap="gray_r")
    plt.xticks(range(n))
    plt.yticks(range(n))
    plt.grid(color="lightgray", linewidth=0.5)

    if path is not None:
        rows = [position[0] for position in path]
        cols = [position[1] for position in path]
        plt.plot(cols, rows, color="red", linewidth=2.5, marker="o", markersize=5, label="Duong di DFS")
        plt.scatter(cols[0], rows[0], c="lime", s=140, edgecolors="black", label="Bat dau")
        plt.scatter(cols[-1], rows[-1], c="yellow", s=140, edgecolors="black", label="Cua ra")

    plt.title("Duong thoat tim duoc bang DFS")
    plt.xlabel("Cot")
    plt.ylabel("Dong")
    plt.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(output_file, dpi=160)
    plt.close()


def main():
    current_dir = Path(__file__).resolve().parent
    input_file = current_dir.parent / "A_in.csv"
    output_file = current_dir / "DFS_out.csv"
    path_image = current_dir / "cau1_dfs_path.png"

    n, start, maze = read_input(input_file)
    path = dfs_escape(n, start, maze)
    write_output(output_file, path)
    save_path_chart(maze, path, path_image)

    if path is None:
        print("DFS khong tim thay duong thoat.")
    else:
        print(f"DFS tim thay duong thoat: {len(path)} o.")
        print(f"Da ghi ket qua vao: {output_file}")
        print(f"Da luu bieu do duong di: {path_image}")


if __name__ == "__main__":
    main()

## Giải thích tác dụng của từng hàm

- `read_input(file_path)`: đọc file CSV đầu vào, lấy kích thước ma trận `n`, vị trí bắt đầu `start` và ma trận lâu đài `maze`.
- `is_inside(position, n)`: kiểm tra một tọa độ có nằm trong phạm vi ma trận `n x n` hay không.
- `is_border(position, n)`: kiểm tra một ô có nằm ở rìa lâu đài hay không. Nếu có, ô đó là cửa ra hợp lệ.
- `get_neighbors(position, n, maze)`: sinh các ô kề cạnh hợp lệ theo bốn hướng trên, phải, dưới, trái. Chỉ những ô nằm trong ma trận và có giá trị `1` mới được đi tiếp.
- `reconstruct_path(parent, goal)`: truy vết đường đi từ ô cửa ra về ô bắt đầu dựa trên dictionary `parent`, sau đó đảo ngược lại để có đường đi đúng chiều.
- `dfs_escape(n, start, maze)`: thực hiện DFS bằng stack. Hàm đi sâu theo từng nhánh, lưu cha của mỗi ô và trả về đường đi khi gặp ô ở rìa.
- `write_output(file_path, path)`: ghi kết quả ra file CSV. Nếu không tìm thấy đường thì ghi `-1`, nếu tìm thấy thì ghi số ô của đường đi và danh sách tọa độ.
- `save_path_chart(maze, path, output_file)`: vẽ ma trận lâu đài và đường đi tìm được, sau đó lưu thành ảnh PNG.
- `main()`: hàm điều phối chính, đọc input, chạy thuật toán, ghi output, lưu biểu đồ và in thông báo kết quả.

## Giải thích thuật toán DFS

DFS là thuật toán tìm kiếm theo chiều sâu. Thay vì duyệt hết các ô gần điểm bắt đầu trước, DFS chọn một nhánh và đi sâu theo nhánh đó cho đến khi gặp cửa ra hoặc không thể đi tiếp.

Trong bài này, mỗi trạng thái là một ô `(row, col)`. DFS dùng `stack` để quản lý các ô chờ xét. Stack hoạt động theo nguyên tắc vào sau ra trước, nên ô được thêm sau sẽ được xử lý trước.

Quy trình thuật toán trong code:

1. Kiểm tra ô bắt đầu `(5,5)` có hợp lệ và đi được không. Nếu không hợp lệ thì trả về `None`.
2. Khởi tạo `stack = [start]`, đưa ô bắt đầu vào stack.
3. Khởi tạo `visited = {start}` để tránh lặp lại các ô đã phát hiện.
4. Khởi tạo `parent = {start: None}` để lưu đường đi từ ô bắt đầu đến từng ô.
5. Trong vòng lặp, lấy ô cuối stack bằng `stack.pop()`.
6. Nếu ô vừa lấy ra nằm trên rìa ma trận, gọi `reconstruct_path()` để dựng lại đường đi và kết thúc.
7. Nếu chưa tới rìa, gọi `get_neighbors()` để lấy các ô kề hợp lệ. Code dùng `reversed(get_neighbors(...))` để khi đưa vào stack, thứ tự xử lý thực tế vẫn đi theo hướng ưu tiên mong muốn.
8. Với mỗi ô kề chưa thăm, đánh dấu `visited`, lưu cha `parent[neighbor] = current`, rồi đưa ô đó vào stack.
9. Nếu stack rỗng mà chưa gặp ô rìa, nghĩa là không tìm thấy đường thoát.

Mô tả vòng lặp DFS:

```text
Lặp khi stack chưa rỗng:
    lấy ô ở cuối stack
    nếu ô đó là rìa -> tìm thấy cửa ra
    nếu chưa -> đẩy các ô kề chưa thăm vào stack
```

DFS có ưu điểm là cách cài đặt đơn giản và có thể tìm lời giải nhanh nếu nhánh đầu tiên dẫn tới cửa ra. Tuy nhiên DFS không bảo đảm đường đi ngắn nhất, vì thuật toán ưu tiên đi sâu theo một nhánh thay vì xét theo từng lớp khoảng cách như BFS. Trong dữ liệu này, DFS tìm được đường đi gồm `9` ô.

## Phân tích chi tiết luồng xử lý

Trước tiên, chương trình kiểm tra ô bắt đầu `(5,5)` có nằm trong ma trận và có đi được không. Nếu ô bắt đầu không hợp lệ thì trả về `None`.

Sau đó thuật toán khởi tạo cấu trúc dữ liệu chính: `stack` chứa các ô chờ xét theo thứ tự LIFO. `visited` lưu các ô đã được đưa vào stack để tránh lặp. `parent` lưu cha của từng ô để truy vết đường đi.

Ở mỗi vòng lặp, thuật toán lấy một ô ra xử lý, kiểm tra ô đó có nằm trên rìa không. Nếu đã ở rìa, chương trình gọi `reconstruct_path()` để dựng lại đường đi. Nếu chưa tới rìa, chương trình gọi `get_neighbors()` để sinh các ô kề hợp lệ rồi đưa các ô phù hợp vào cấu trúc chờ xét.

## Bảng bước chạy chi tiết

| Bước | Ô lấy ra | Thêm vào stack | Stack sau bước | Ghi chú |
| --- | --- | --- | --- | --- |
| 1 | `(5,5)` | `(5,6)`, `(4,5)` | `[(5,6), (4,5)]` | Tiếp tục |
| 2 | `(4,5)` | `(3,5)` | `[(5,6), (3,5)]` | Tiếp tục |
| 3 | `(3,5)` | `(3,4)` | `[(5,6), (3,4)]` | Tiếp tục |
| 4 | `(3,4)` | `(3,3)` | `[(5,6), (3,3)]` | Tiếp tục |
| 5 | `(3,3)` | `(2,3)` | `[(5,6), (2,3)]` | Tiếp tục |
| 6 | `(2,3)` | `(1,3)` | `[(5,6), (1,3)]` | Tiếp tục |
| 7 | `(1,3)` | `(1,2)` | `[(5,6), (1,2)]` | Tiếp tục |
| 8 | `(1,2)` | `(1,1)`, `(0,2)` | `[(5,6), (1,1), (0,2)]` | Tiếp tục |
| 9 | `(0,2)` |  | `[(5,6), (1,1)]` | Gặp cửa ra |

## Biểu đồ đường đi

![Đường thoát tìm được](cau1_dfs_path.png)

Trong hình minh họa:

- Ô có giá trị `1` là ô đi được.
- Ô có giá trị `0` là tường hoặc không có đường hầm.
- Đường màu đỏ nối các tọa độ trong danh sách đường đi.
- Điểm đầu là `(5,5)`, tức phòng trung tâm.
- Điểm cuối là ô nằm trên rìa ma trận, tức cửa ra.

Đường đi trong file output là:

`(5,5)` -> `(4,5)` -> `(3,5)` -> `(3,4)` -> `(3,3)` -> `(2,3)` -> `(1,3)` -> `(1,2)` -> `(0,2)`

Dòng đầu của file output là `9`, nghĩa là đường đi có `9` ô. Nếu tính số bước di chuyển thì số bước bằng `9 - 1`.

## Kết quả thực thi

Lệnh chạy chương trình:

```powershell
python 2025/De1/BFS_DFS_Astar_Manhattan_Cau1/03_DFS/cau1_dfs.py
```

Kết quả trong `DFS_out.csv`:

```text
9
5,5
4,5
3,5
3,4
3,3
2,3
1,3
1,2
0,2
```

Kết luận: DFS tìm được đường thoát hợp lệ từ `(5,5)` đến một ô ở rìa ma trận.